In [1]:
"""
This scripts accurately produces qPCR Cqs from template. N=5. Mixing in col=1
Aliquot 16*6 = 96ul qPCR Mix into 1st col; 4*6=24ul template into same well. 
Mixes, then adds to wells A2..A5. 

Author : Harley King
Date   : 2025-08-29


This program works well. Don't forget to manually change pipette tip columns down below. Four places in the code!

"""


"\nThis scripts accurately produces qPCR Cqs from template. N=5. Mixing in col=1\nAliquot 16*6 = 96ul qPCR Mix into 1st col; 4*6=24ul template into same well. \nMixes, then adds to wells A2..A5. \n\nAuthor : Harley King\nDate   : 2025-08-29\n\n\nThis program works well. Don't forget to manually change pipette tip columns down below. Four places in the code!\n\n"

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import asyncio
from typing import List, Iterator

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources.hamilton import STARLetDeck, MFX_CAR_L5_base, TIP_CAR_480_A00
from pylabrobot.resources.hamilton.mfx_modules import Hamilton_MFX_plateholder_DWP_metal_tapped
from pylabrobot.resources.opentrons.tube_racks import (
    opentrons_24_tuberack_generic_1point5ml_snapcap_short,
)
from pylabrobot.resources.tube_adapter import TubeRackAdapter
from pylabrobot.resources.agenbio.plates import AGenBio_1_troughplate_100000uL_Fl
from pylabrobot.resources.bioer.plates import BioER_96_wellplate_Vb_2200ul
from pylabrobot.resources import (
    TIP_50ul_w_filter, # 50 µL filtered
    HTF,     # 1000 µL filtered 
    LTF #Tip Rack with 96 10ul Low Volume Tip with filter
)

###############################################################################
# 0) build LiquidHandler + deck
###############################################################################
backend = STARBackend()
lh      = LiquidHandler(backend=backend, deck=STARLetDeck())
# await lh.stop()
await lh.setup(skip_autoload=True)

In [4]:
from pylabrobot.resources.plate import Plate
from pylabrobot.resources.utils import create_ordered_items_2d
from pylabrobot.resources.well import (
  CrossSectionType,
  Well,
  WellBottomType,
)
def VWR_96_wellplate_100_Vb(name: str, with_lid: bool = False) -> Plate:
  """
This plate is a VWR PCR plate 96 well low-profile, half-skirted, ABI-FAST type plate.
VWR cat no. 89218-296
It is half-skirted so it must reside in another plate like a Cor_96_wellplate_360ul_Fb
  """
  
  return Plate(
    name=name,
    size_x=127.76,
    size_y=85.48,
    size_z=20.0,
    # lid=lid,
    model=VWR_96_wellplate_100_Vb.__name__,
    ordered_items=create_ordered_items_2d(
      Well,
      num_items_x=12,
      num_items_y=8,
      dx=11.5,  # keeping costar measurement. Previously 10.25
      dy=8.75,  # 7.77 keeping costar measurement. Previoulsy 10.5, 11, 
      dz=8.5, # how high is well above base
      item_dx=9.0,
      item_dy=9.0,
      size_x=5.4,  # measured
      size_y=5.4,  # measured
      size_z=16.3, # measured well depth, costar + VWR plate height
      material_z_thickness=0.5,
      bottom_type=WellBottomType.V,
      cross_section_type=CrossSectionType.CIRCLE,
      max_volume=100,
    ),
  )

from typing import Optional

from pylabrobot.resources.height_volume_functions import (
  compute_height_from_volume_rectangle,
  compute_volume_from_height_rectangle,
)
from pylabrobot.resources.plate import Lid, Plate
from pylabrobot.resources.utils import create_ordered_items_2d
from pylabrobot.resources.well import (
  CrossSectionType,
  Well,
  WellBottomType,
)

In [5]:
###############################################################################
# 1) carriers, modules & labware
###############################################################################
# --- tip carrier -------------------------------------------------------------
# --- tip carrier -------------------------------------------------------------
tip_car = TIP_CAR_480_A00("tip_car")
lh.deck.assign_child_resource(tip_car, rails=25)
tiprack_1000 = HTF("tips_00")              # 1000 µL filter tips (slot-0)
tiprack_50   = TIP_50ul_w_filter("tips_01") #  50 µL filter tips (slot-1)
tiprack_10 = LTF("tips_02") #10 ul filter tips
# mount the racks
tip_car[0] = tiprack_1000          # OR:  tip_car[0].assign_child_resource(tiprack_1000)
tip_car[1] = tiprack_50
tip_car[2] = tiprack_10


# STANDARDS RACK
dwp_mod_dest   = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_dest")
car_19 = MFX_CAR_L5_base(
    "car_19",
    modules={
        0: dwp_mod_dest
    }
)
lh.deck.assign_child_resource(car_19, rails=19)

# # labwareopentrons_24_tuberack_generic_1point5ml_snapcap_short
dw_dilutions = BioER_96_wellplate_Vb_2200ul("dw_dilutions")
# dest_offset_x = (127.76 - tuberack_dest._size_x) / 2
# dest_offset_y = (85.48  - tuberack_dest._size_y) / 2

# adapter_dest = TubeRackAdapter(
#     name="dest_rack_adapter",
#     size_x=127.76,
#     size_y=85.48,
#     size_z=tuberack_dest._size_z,              # external height of the frame
#     model="tube_rack_adapter",
#     dx=dest_offset_x,
#     dy=dest_offset_y,
#     dz=0,
#     adapter_hole_size_x=tuberack_dest._size_x,
#     adapter_hole_size_y=tuberack_dest._size_y,
#     adapter_hole_size_z=tuberack_dest._size_z
# )
dwp_mod_dest.assign_child_resource(dw_dilutions)

# ----------carrier @ rail 13: water trough-------------------
# 96W, 100ul VWR PCR plate
dwp_mod_PCR = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_PCR")
dwp_mod_trough = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_trough")
car_13 = MFX_CAR_L5_base(
    "car_13",
    modules={
        0: dwp_mod_PCR,
        1: dwp_mod_trough,
    }
)
lh.deck.assign_child_resource(car_13, rails=13)
qPCR_plate = VWR_96_wellplate_100_Vb("qPCR_plate")
dwp_mod_PCR.assign_child_resource(qPCR_plate)
trough = AGenBio_1_troughplate_100000uL_Fl("water_trough")
dwp_mod_trough.assign_child_resource(trough)

# --- carrier @ rail-7: OT-2 tube rack with dsDNA ----------------------------
dwp_mod_src = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_src")
car_07 = MFX_CAR_L5_base(
    "car_07",
    modules={
        0: dwp_mod_src,
    }
)
lh.deck.assign_child_resource(car_07, rails=7)

tuberack_src = opentrons_24_tuberack_generic_1point5ml_snapcap_short("src_rack")
src_offset_x = (127.76 - tuberack_src._size_x) / 2
src_offset_y = (85.48  - tuberack_src._size_y) / 2

adapter_src = TubeRackAdapter(
    name="src_rack_adapter",
    size_x=127.76,
    size_y=85.48,
    size_z=tuberack_src._size_z,              # external height of the frame
    model="tube_rack_adapter",
    dx=src_offset_x,
    dy=src_offset_y,
    dz=0,
    adapter_hole_size_x=tuberack_src._size_x,
    adapter_hole_size_y=tuberack_src._size_y,
    adapter_hole_size_z=tuberack_src._size_z
)
adapter_src.assign_child_resource(tuberack_src)
dwp_mod_src.assign_child_resource(adapter_src)

In [ ]:
CHANNEL_MM   = 6            # single channel we’ll use for the whole run
TIPRACK_50   = tiprack_50   # 50 µL filter tips (slot-1 on tip_car)

SRC_MM_TUBE  = tuberack_src["D6"]   # mastermix (carrier @ rail 7, tube rack “src_rack”)
DEST_PLATE   = qPCR_plate           # 96-well PCR plate (carrier @ rail 13)
START_TIP = "A1"
STANDARDS_RACK = dw_dilutions      # dilution-series rack (carrier @ rail 19)


ROW_LET = "ABCDEFGH"

CHANNELS_8  = list(range(8))

# If you already have tiprack_1000 in scope, this alias is optional:
TIPRACK_1000 = tiprack_1000

# -------- 1) Mastermix into A1..H1 and A6..H6 (single channel) --------
async def dispense_mastermix():
    # one 1000 µL tip (reused for both columns)
    await lh.pick_up_tips(TIPRACK_1000[START_TIP], use_channels=[CHANNEL_MM])

    for col in (1, 7):
        # Pre-wet + slow handling for viscous MM
        try:
            await lh.aspirate(
                SRC_MM_TUBE,
                vols=[900],
                use_channels=[CHANNEL_MM],
                mix_volume=[900],
                mix_cycles=[1],
                mix_speed = [150], # 50 too slow
                mix_surface_following_distance=[14],
                # pre_wetting_volume=[900], # this errs at [900]
                settling_time=[2],
                lld_mode=[STARBackend.LLDMode.GAMMA],
                immersion_depth=[2],
                immersion_depth_direction=[0],
                surface_following_distance=[14],
                transport_air_volume=[0],
                # flow_rates=[40],  # uncomment if your backend supports this param
            )
        except: # if not enought mmix
            await lh.aspirate(
                SRC_MM_TUBE,
                vols=[900],
                use_channels=[CHANNEL_MM],
                liquid_height=[1],
                settling_time=[2],
                transport_air_volume=[0]
            )
        # Condition the tip
    
        try:
            await lh.dispense(
                SRC_MM_TUBE, vols=[50],
                use_channels=[CHANNEL_MM],
                lld_mode=[STARBackend.LLDMode.GAMMA],
                settling_time=[2],
                transport_air_volume=[0],
                # flow_rates=[40],
            )
        except: # if liquid level too low GAMMA throws an error
            await lh.dispense(
                SRC_MM_TUBE, vols=[50],
                use_channels=[CHANNEL_MM],
                liquid_height=[1],
                settling_time=[2],
                transport_air_volume=[0],
                # flow_rates=[40],
            )
        # Walk down the column A..H at this col; 100 µL each
        for i, row in enumerate(ROW_LET):
            last = (i == len(ROW_LET) - 1)
            await lh.dispense(
                DEST_PLATE[f"{row}{col}"],
                vols=[100],
                use_channels=[CHANNEL_MM],
                liquid_height=[6],
                transport_air_volume=[0],
                settling_time=[2],
                blow_out=[1] if last else [0]       # clear any remainder at the last well
            )

    await lh.discard_tips()


# -------- 2) Add 24 µL template from deepwell to qPCR plate (8-ch) --------
async def dispense_template():
    # A1..H1 → qPCR A1..H1
    await lh.pick_up_tips(TIPRACK_50["A1:H1"], use_channels=CHANNELS_8)
    await lh.aspirate(
        STANDARDS_RACK["A1:H1"],
        vols=[24]*8,
        use_channels=CHANNELS_8,
        lld_mode=[STARBackend.LLDMode.GAMMA]*8,
        mix_volume=[30]*8,
        mix_cycles=[2]*8,
        mix_surface_following_distance=[2]*8,
        surface_following_distance=[2]*8,
        transport_air_volume=[0]*8,
        flow_rates=[10]*8,    
        immersion_depth=[1]*8
    )
    await lh.dispense(
        DEST_PLATE["A1:H1"],
        vols=[24]*8,
        use_channels=CHANNELS_8,
        lld_mode=[STARBackend.LLDMode.GAMMA]*8,
        immersion_depth=[1]*8,                 # ~1 mm under surface to avoid bubbles
        immersion_depth_direction=[0]*8,
        # transport_air_volume=[0]*8, # don't want to risk falling into nearby wells
        flow_rates=[10]*8,
        blow_out=[1]*8
    )
    await lh.discard_tips()

    # A6..H6 → qPCR A6..H6
    await lh.pick_up_tips(TIPRACK_50["A2:H2"], use_channels=CHANNELS_8)
    await lh.aspirate(
        STANDARDS_RACK["A6:H6"],
        vols=[24]*8,
        use_channels=CHANNELS_8,
        lld_mode=[STARBackend.LLDMode.GAMMA]*8,
        mix_volume=[30]*8,
        mix_cycles=[2]*8,
        mix_surface_following_distance=[2]*8,
        transport_air_volume=[0]*8,
        immersion_depth=[1]*8,
        flow_rates=[10]*8
    )
    await lh.dispense(
        DEST_PLATE["A7:H7"],
        vols=[24]*8,
        use_channels=CHANNELS_8,
        lld_mode=[STARBackend.LLDMode.GAMMA]*8,
        immersion_depth=[1]*8,
        immersion_depth_direction=[0]*8,
        transport_air_volume=[0]*8,
        flow_rates=[10]*8,
        blow_out=[1]*8
    )
    await lh.discard_tips()


# -------- 3) Mix in source columns and aliquot forward (8-ch) --------
async def mix_and_aliquot():
    # Use A3:H3 once to spread column 1 → cols 2..6
    await lh.pick_up_tips(TIPRACK_50["A3:H3"], use_channels=CHANNELS_8)
    # mixing the mix + template homogenously
    await lh.aspirate(
        DEST_PLATE["A1:H1"],
        vols=[0]*8,
        use_channels=CHANNELS_8,
        lld_mode=[STARBackend.LLDMode.GAMMA]*8,
        mix_volume=[50]*8,
        mix_cycles=[15]*8,
        mix_surface_following_distance=[4]*8,
        surface_following_distance = [2]*8,
        transport_air_volume=[0]*8,
        settling_time=[1]*8,
        immersion_depth=[1]*8,
        blow_out=[1]*8
    )
    for col in range(2, 7):  # 2,3,4,5,6
        # Mix at A1..H1, then take 20 µL
        
        await lh.aspirate(
            DEST_PLATE["A1:H1"],
            vols=[23]*8,
            use_channels=CHANNELS_8,
            lld_mode=[STARBackend.LLDMode.GAMMA]*8,
            mix_volume=[50]*8,
            mix_cycles=[2]*8,
            mix_surface_following_distance=[4]*8,
            surface_following_distance = [2]*8,
            transport_air_volume=[0]*8,
            settling_time=[1]*8,
            immersion_depth=[1]*8
        )
        await lh.dispense(
            DEST_PLATE[f"A{col}:H{col}"],
            vols=[23]*8,
            use_channels=CHANNELS_8,
            liquid_height=[2]*8,                 # target ~2 mm above bottom in dest
            transport_air_volume=[0]*8,
            flow_rates=[10]*8,
            settling_time=[1]*8,
            blow_out=[1]*8
        )
    await lh.discard_tips()

    # New tips A4..H4 to spread column 6 → cols 7..12 (H6 stays unused as requested)
    await lh.pick_up_tips(TIPRACK_50["A4:H4"], use_channels=CHANNELS_8)
    # homogenize mix + template 
    await lh.aspirate(
    DEST_PLATE["A7:H7"],
    vols=[0]*8,
    use_channels=CHANNELS_8,
    lld_mode=[STARBackend.LLDMode.GAMMA]*8,
    mix_volume=[50]*8,
    mix_cycles=[15]*8,
    mix_surface_following_distance=[4]*8,
    surface_following_distance = [2]*8,
    transport_air_volume=[0]*8,
    settling_time=[1]*8,
    immersion_depth=[1]*8,
    blow_out=[1]*8
    )
    for col in range(8, 13):  # 7..12
        await lh.aspirate(
            DEST_PLATE["A7:H7"],
            vols=[23]*8,
            use_channels=CHANNELS_8,
            lld_mode=[STARBackend.LLDMode.GAMMA]*8,
            mix_volume=[50]*8,
            mix_cycles=[2]*8,
            mix_surface_following_distance=[4]*8,
            transport_air_volume=[0]*8,
            surface_following_distance = [2]*8,
            settling_time=[1]*8,
            immersion_depth=[1]*8
        )
        await lh.dispense(
            DEST_PLATE[f"A{col}:H{col}"],
            vols=[23]*8,
            use_channels=CHANNELS_8,
            liquid_height=[2]*8,
            transport_air_volume=[0]*8,
            flow_rates=[10]*8,
            settling_time=[1]*8,
            blow_out=[1]*8
        )
    await lh.discard_tips()


In [ ]:
await dispense_mastermix()
await dispense_template()
await mix_and_aliquot()

In [ ]:
# await lh.dispense(trough["A1"], vols=[900], liquid_height=[2], use_channels=[CHANNEL_WATER])
# await lh.dispense(SRC_MM_TUBE, vols=[100], liquid_height=[4], use_channels=[CHANNEL_MM], blow_out=[1])
# await lh.dispense(DEST_PLATE["A7:H7"], vols=[50], liquid_height=[7], use_channels=CHANNELS_8, blow_out=[1])
# await lh.drop_tips(TIPRACK_50["A5:H5"], use_channels=CHANNELS_8)
# await lh.drop_tips(tiprack_1000["A1"], use_channels=[2])
# await lh.discard_tips()
# await lh.stop()
# await backend.stop() 

# await lh.dispense(
#     STANDARDS_RACK["A1:H1"],
#     vols=[24]*8,
#     use_channels=CHANNELS_8,
#     lld_mode=[STARBackend.LLDMode.GAMMA]*8,
#     blow_out=[1]*8
# )

await lh.dispense(
        DEST_PLATE["A7:H7"],
        vols=[23]*8,
        use_channels=CHANNELS_8,
        liquid_height=[2]*8,
        transport_air_volume=[0]*8,
        flow_rates=[10]*8,
        settling_time=[1]*8,
        blow_out=[1]*8
    )

In [ ]:
await lh.discard_tips()
